In [2]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

# =========================================================
# SELECCIÓN DEL PESO w ÓPTIMO DEL MLP-VaR
# Periodo de validación: 2010-2014 (fuera de muestra de test)
# Pesos candidatos: w ∈ {1, 5, 10, 15, 20, 25}
#
# Criterio de selección (mismo que en comp_final):
#   1. Validez: pasa simultáneamente CC y DQ (p > 0.05)
#   2. Menor VR Gap  (calibración en frecuencia)
#   3. Menor Tick Loss (calibración en cuantil)
# =========================================================

W_LIST = [1, 5, 10, 15, 20, 25]

In [3]:
# =========================================================
# FRAMEWORK DE MÉTRICAS (idéntico a comp_final)
# =========================================================

def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds   = np.asarray(var_preds,   dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def exceedance_stats(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, I = hit_series(real_losses, var_preds)
    T  = len(I)
    x  = int(I.sum())
    p  = 1 - alpha
    rv = x / T if T > 0 else np.nan
    return {
        "T": T,
        "Expected Viol.": p * T,
        "Violations": x,
        "Violation Rate": rv,
        "VR Gap": abs(rv - p) if T > 0 else np.nan,
        "Coverage Bias": (rv - p) if T > 0 else np.nan,
    }


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over  = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def mean_var_level(var_preds):
    return float(np.nanmean(np.asarray(var_preds, dtype=float)))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x   = len(I), int(I.sum())
    p      = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc  = -2 * np.log(
        ((1 - p) ** (T - x) * p ** x) /
        ((1 - p_hat) ** (T - x) * p_hat ** x)
    )
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if   pair == (0, 0): n00 += 1
        elif pair == (0, 1): n01 += 1
        elif pair == (1, 0): n10 += 1
        else:                n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi   = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup  = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or (LRind is np.nan) or np.isnan(LRind):
        LRcc, p_cc = np.nan, np.nan
    else:
        LRcc = LRuc + LRind
        p_cc = 1 - chi2.cdf(LRcc, df=2)
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": p_cc}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y   = hit[lags:]
    X   = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta  = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ   = float((beta.T @ XtX @ beta) / sigma2)
    p_dq = 1 - chi2.cdf(DQ, df=X.shape[1])
    return {"DQ": DQ, "p_dq": p_dq}


def evaluate_var_model(real_losses, var_preds, alpha=0.99):
    exc = exceedance_stats(real_losses, var_preds, alpha=alpha)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc  = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq  = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    tl  = tick_loss(real_losses, var_preds, alpha=alpha)
    ws  = winkler_score(real_losses, var_preds, alpha=alpha)
    mvl = mean_var_level(var_preds)
    pvm = float(np.nanmean([v for v in [kup["p_uc"], cc["p_cc"], dq["p_dq"]] if pd.notnull(v)]))
    return {
        "T":              exc["T"],
        "Expected Viol.": exc["Expected Viol."],
        "Violations":     exc["Violations"],
        "Violation Rate": exc["Violation Rate"],
        "VR Gap":         exc["VR Gap"],
        "Coverage Bias":  exc["Coverage Bias"],
        "Tick Loss":      tl,
        "Winkler Score":  ws,
        "Mean VaR Level": mvl,
        "p_UC":  kup["p_uc"],
        "p_CC":  cc["p_cc"],
        "p_DQ":  dq["p_dq"],
        "p_Mean": pvm,
    }

In [4]:
# =========================================================
# CARGA DE PREDICCIONES DE VALIDACIÓN Y EVALUACIÓN
# =========================================================

rows = []
for w in W_LIST:
    df = pd.read_csv(f"../data/nn_val_predictions_{w}.csv", index_col=0, parse_dates=True)
    res = evaluate_var_model(df["loss_real"].values, df["VaR_pred"].values)
    res["w"] = w
    rows.append(res)

results = pd.DataFrame(rows)

SIG = 0.05
results["UC Test"] = np.where(results["p_UC"] > SIG, "PASS", "FAIL")
results["CC Test"] = np.where(results["p_CC"] > SIG, "PASS", "FAIL")
results["DQ Test"] = np.where(results["p_DQ"] > SIG, "PASS", "FAIL")
results["Valid (CC&DQ)"] = np.where(
    (results["CC Test"] == "PASS") & (results["DQ Test"] == "PASS"), "YES", "NO"
)
results["_rank"] = np.where(results["Valid (CC&DQ)"] == "YES", 0, 1)

results = results.sort_values(["_rank", "VR Gap", "Tick Loss"]).reset_index(drop=True)

SEP = "=" * 110
sep = "-" * 110

print(f"\n{SEP}")
print("SELECCIÓN DEL PESO w — MLP-VaR | Periodo de validación: 2010-2014")
print(SEP)

disp_cols = [
    "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level",
    "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test",
    "p_Mean", "Valid (CC&DQ)"
]

show = results[disp_cols].copy()
fmt = {
    "Expected Viol.": lambda x: f"{x:.2f}",
    "Violation Rate": lambda x: f"{x:.4f}",
    "VR Gap":         lambda x: f"{x:.4f}",
    "Coverage Bias":  lambda x: f"{x:+.4f}",
    "Tick Loss":      lambda x: f"{x:.6f}",
    "Winkler Score":  lambda x: f"{x:.6f}",
    "Mean VaR Level": lambda x: f"{x:.4f}",
    "p_UC":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_CC":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_DQ":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_Mean":         lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
}
print(show.to_string(index=False, formatters=fmt))


SELECCIÓN DEL PESO w — MLP-VaR | Periodo de validación: 2010-2014
 w    T Expected Viol.  Violations Violation Rate VR Gap Coverage Bias Tick Loss Winkler Score Mean VaR Level  p_UC UC Test  p_CC CC Test  p_DQ DQ Test p_Mean Valid (CC&DQ)
25 1256          12.56          16         0.0127 0.0027       +0.0027  0.001064      0.088162         0.0878 0.349    PASS 0.525    PASS 0.896    PASS  0.590           YES
20 1256          12.56          16         0.0127 0.0027       +0.0027  0.001065      0.087903         0.0875 0.349    PASS 0.525    PASS 0.896    PASS  0.590           YES
15 1256          12.56          17         0.0135 0.0035       +0.0035  0.001064      0.087499         0.0871 0.232    PASS 0.388    PASS 0.479    PASS  0.366           YES
10 1256          12.56          18         0.0143 0.0043       +0.0043  0.001078      0.086778         0.0863 0.147    PASS 0.269    PASS 0.461    PASS  0.293           YES
 5 1256          12.56          19         0.0151 0.0051       +0.00

In [5]:
# =========================================================
# VEREDICTO: w GANADOR
# =========================================================

valid = results[results["Valid (CC&DQ)"] == "YES"]

print(f"\n{SEP}")
print("CRITERIO DE SELECCIÓN")
print(SEP)
print("  Prioridad 1 : supera simultáneamente CC y DQ (p > 0.05 en ambos).")
print("  Prioridad 2 : menor VR Gap  (calibración en frecuencia).")
print("  Prioridad 3 : menor Tick Loss (calibración en cuantil).")

if len(valid) > 0:
    best = valid.sort_values(["VR Gap", "Tick Loss"]).iloc[0]
    w_best = int(best["w"])
    print(f"\n  PESO SELECCIONADO: w = {w_best}")
    print(f"\n  Razón:")
    print(f"    — Supera CC y DQ con p_CC={best['p_CC']:.3f} y p_DQ={best['p_DQ']:.3f}.")
    print(f"    — VR Gap = {best['VR Gap']:.4f}  (tasa de violación {best['Violation Rate']:.4f} frente al 1% teórico).")
    print(f"    — Tick Loss = {best['Tick Loss']:.6f}.")
else:
    best = results.sort_values(["VR Gap", "Tick Loss"]).iloc[0]
    w_best = int(best["w"])
    print(f"\n  Ningún w supera simultáneamente CC y DQ en validación.")
    print(f"  PESO SELECCIONADO (por menor VR Gap y Tick Loss): w = {w_best}")

print(f"\n{SEP}")
print(f"  CONCLUSIÓN METODOLÓGICA")
print(SEP)
print(f"""  El peso w = {w_best} es seleccionado a partir del periodo de validación 2010-2014,""")
print(f"""  que es independiente del periodo de evaluación final (2015-2024).""")
print(f"""  Este enfoque evita el look-ahead bias que se produciría al elegir w""")
print(f"""  observando los resultados del propio conjunto de test.""")
print(f"""  Los resultados reportados en comp_final_{w_best}.ipynb corresponden al modelo""")
print(f"""  seleccionado por este procedimiento riguroso de validación.""")
print(SEP)


CRITERIO DE SELECCIÓN
  Prioridad 1 : supera simultáneamente CC y DQ (p > 0.05 en ambos).
  Prioridad 2 : menor VR Gap  (calibración en frecuencia).
  Prioridad 3 : menor Tick Loss (calibración en cuantil).

  PESO SELECCIONADO: w = 25

  Razón:
    — Supera CC y DQ con p_CC=0.525 y p_DQ=0.896.
    — VR Gap = 0.0027  (tasa de violación 0.0127 frente al 1% teórico).
    — Tick Loss = 0.001064.

  CONCLUSIÓN METODOLÓGICA
  El peso w = 25 es seleccionado a partir del periodo de validación 2010-2014,
  que es independiente del periodo de evaluación final (2015-2024).
  Este enfoque evita el look-ahead bias que se produciría al elegir w
  observando los resultados del propio conjunto de test.
  Los resultados reportados en comp_final_25.ipynb corresponden al modelo
  seleccionado por este procedimiento riguroso de validación.
